In [ ]:
'''libraries'''
#Data
import pandas as pd

#Plots
import matplotlib.pyplot as plt

#math
import numpy as np

#Constants
from scipy.constants import physical_constants
m_u=physical_constants['atomic mass constant energy equivalent in MeV'][0]
from scipy.constants import speed_of_light as c

#Usefull
from tqdm.notebook import tqdm
import os
from scipy.interpolate import interp1d
import pynucastro as pyna
#%matplotlib widget

Read Data from the Snaps


In [ ]:
''''''
#Run1
run_rates={'reactions': "Reaclib_18_9_20", 'Beta':  "Langanke, Martinez-Pinedo",  'Alpha': "the Viola-Seaborg formula"}
input_rates=r'Nuclear_Data\run1\Reaclib_18_9_20'
input_masses=r'Nuclear_Data\run1\winvne_v2.0.dat'
folder = r'Runs\run1\snaps'

#Run2
run_rates={'reactions': "Reaclib_18_9_20", 'Beta':  "None",  'Alpha': "None"}
input_rates=r'Nuclear_Data\run2\Reaclib_18_9_20'
input_masses=r'Nuclear_Data\run2\winvne_v2.0.dat'
folder = r'Runs\run2\snaps'

#Run3
run_rates={'reactions': "Reaclib_exp_R1", 'Beta':  "None",  'Alpha': "None"}
input_rates=r'Nuclear_Data\run3\Reaclib_exp_R1'
input_masses=r'Nuclear_Data\run1\winvne_v2.0.dat'
folder = r'Runs\run3\snaps'

#Run4
run_rates={'reactions': "Reaclib_NO_T", 'Beta':  "Langanke, Martinez-Pinedo",  'Alpha': "the Viola-Seaborg formula"}
input_rates=r'Nuclear_Data\run4\Reaclib_NO_T'
input_masses=r'Nuclear_Data\run1\winvne_v2.0.dat'
folder = r'Runs\run4\snaps'

#Run5
run_rates={'reactions': "Reaclib_NO_T", 'Beta':  "None",  'Alpha': "None"}
input_rates=r'Nuclear_Data\run4\Reaclib_NO_T'
input_masses=r'Nuclear_Data\run1\winvne_v2.0.dat'
folder = r'Runs\run5\snaps'

#Run6
run_rates={'reactions': "Reaclib_pynucastro_R1", 'Beta':  "None",  'Alpha': "None"}
input_rates=r'Nuclear_Data\run6\Reaclib_pynucastro_R1'
input_masses=r'Nuclear_Data\run1\winvne_v2.0.dat'
folder = r'Runs\run6\snaps'


files = os.listdir(folder)

abundances_winnet=[]
time_Winnet=[]


for file in tqdm(files):
    time_i=float(np.loadtxt(os.path.join(folder, file), skiprows=1,max_rows=1, usecols=0))
    if time_i>1:
        abundances_winnet.append(np.loadtxt(os.path.join(folder, file), skiprows=3))
        time_Winnet.append(float(np.loadtxt(os.path.join(folder, file), skiprows=1,max_rows=1, usecols=0)))

abundances_winnet = np.array(abundances_winnet)
time_Winnet = np.array(time_Winnet) 
nuclear_data=pd.read_csv(input_masses,skiprows=lambda x: not (x>7854 and (x-7855)%4==0),delim_whitespace=True,names=['name','A','Z','N','spin','Mass excess (Mev)','source'])


Interpolation to have a smooth Xi curve

In [ ]:
initial_time=time_Winnet[0] #seconds
final_time=4.32E7 #seconds 500 days= 4.32E7 seconds 100 days= 8.64E7 seconds
N_steps=10000
time = np.exp(np.linspace(np.log(initial_time), np.log(final_time), N_steps))
abundances_time= np.empty((abundances_winnet.shape[1],N_steps))

for i in tqdm(range(abundances_winnet.shape[1])):
    spline_interp = interp1d(time_Winnet, abundances_winnet[:,i,3], kind='linear')
    abundances_time[i] = spline_interp(time)


def index_n_z(n,z):
    idx_start = np.searchsorted(abundances_winnet[0,:,1], z, side='left')
    idx_end = np.searchsorted(abundances_winnet[0,:,1], z, side='right')
    if idx_end>idx_start:
        j=np.searchsorted(abundances_winnet[0,:,0][idx_start:idx_end],n)+idx_start
        if abundances_winnet[0,:,0][j]==n and abundances_winnet[0,:,1][j]==z:
            return j
        else:
            return 'None'
    else:
        return 'None'
    
def Xi_time_n_z(n,z):

    j=index_n_z(n,z)
    if j=='None':
        return np.zeros(N_steps)
    elif abundances_winnet[0,:,0][j]==n and abundances_winnet[0,:,1][j]==z:
        return abundances_time[j]
    else:
        return np.zeros(N_steps)
    
def index_winv_v2(z,n):
        
    idx_start = np.searchsorted(nuclear_data['Z'], z, side='left')
    idx_end = np.searchsorted(nuclear_data['Z'], z, side='right')
    if idx_end>idx_start:
        j=np.searchsorted(nuclear_data['N'][idx_start:idx_end],n)+idx_start
        if nuclear_data['N'][j]==n and nuclear_data['Z'][j]==z:
            return j
        else:
            return 'None'
    else:
        return 'None'
    

Rates used in the simulation

In [ ]:
rates_Reaclib_winnet=pyna.rates.library.Library(
    libfile=input_rates
    )

if run_rates['Beta']!=None:
    if run_rates['Beta']=='Langanke, Martinez-Pinedo':
        weak_rates_file='Nuclear_Data/Beta_tabulated/LMP2001.dat'
if run_rates['Alpha']!=None:
    if run_rates['Alpha']=='the Viola-Seaborg formula':
        alpha_decay_file='Nuclear_Data/Alpha_tabulated/VS.dat'


# Remove duplicate links from the library

for pair in rates_Reaclib_winnet.find_duplicate_links():
    if pair[0].eval_deriv(1e9)==0:
        rates_Reaclib_winnet.remove_rate(pair[1])
    else:
        rates_Reaclib_winnet.remove_rate(pair[0])


filter_alpha=pyna.RateFilter(
    products=['he4'],
    exact=False,
    max_reactants=1,
    max_products=2,
    filter_function=lambda r: r.Q>0 and r.reactants[0].Z==r.products[0].Z+r.products[1].Z)

rates_alpha=rates_Reaclib_winnet.filter(filter_alpha)

filter_beta_minus=pyna.RateFilter(
    max_reactants=1,
    max_products=1,
    filter_function=lambda r: r.Q>0 and r.reactants[0].Z==r.products[0].Z-1)

rates_beta_minus=rates_Reaclib_winnet.filter(filter_beta_minus)

print('number of alpha decays '+str(len(rates_alpha.get_rates()))+', number of beta decays '+str(len(rates_beta_minus.get_rates())))


Calcualtion of $\epsilon (t)$

In [ ]:

e_b=np.zeros(N_steps)
e_a=np.zeros(N_steps)
e_b_nuclei_erg=[]
e_a_nuclei_erg=[]
nuclei_b=[]
nuclei_a=[]
n=0
nb=0
na=0


for beta_rate in tqdm(rates_beta_minus.get_rates()):
    nuclei=beta_rate.reactants[0]
    Z_i=nuclei.Z
    N_i=nuclei.N
    m_i=0
    if type(nuclei.dm)==float:
        m_i=nuclei.dm+(Z_i+N_i)*m_u
        decay_rate_i=beta_rate.eval(1e9)
        Q_i=beta_rate.Q
        X_i=Xi_time_n_z(N_i,Z_i)
        e=decay_rate_i*(Q_i*X_i*c**2)/m_i
        e_b+=e
        e_b_nuclei_erg.append(e*1e4)
        nuclei_b.append(nuclei)
    elif index_winv_v2(nuclei.Z,nuclei.N)!='None':
        m_i=nuclear_data['Mass excess (Mev)'][index_winv_v2(Z_i,N_i)]+(Z_i+N_i)*m_u
        decay_rate_i=beta_rate.eval(1e9)
        Q_i=beta_rate.Q
        X_i=Xi_time_n_z(N_i,Z_i)
        e=decay_rate_i*(Q_i*X_i*c**2)/m_i
        e_b+=e
        e_b_nuclei_erg.append(e*1e4)
        nuclei_b.append(nuclei)
    else:
        nb+=1
    
for alpha_rate in tqdm(rates_alpha.get_rates()):
    nuclei=alpha_rate.reactants[0]
    Z_i=nuclei.Z
    N_i=nuclei.N
    if type(nuclei.dm)==float:
        if index_n_z(N_i,Z_i)!='None':
            decay_rate_i=alpha_rate.eval(1e9)
            Q_i=alpha_rate.Q
            m_i=nuclei.dm+(Z_i+N_i)*m_u
            X_i=Xi_time_n_z(N_i,Z_i)
            e=decay_rate_i*(Q_i*X_i*c**2)/m_i
            e_a+=e
            e_a_nuclei_erg.append(e*1e4)
            nuclei_a.append(nuclei)     
           
        else: 
            n+=1
    elif index_winv_v2(nuclei.Z,nuclei.N)!='None':
        if index_n_z(N_i,Z_i)!='None':
            decay_rate_i=alpha_rate.eval(1e9)
            Q_i=alpha_rate.Q
            m_i=nuclear_data['Mass excess (Mev)'][index_winv_v2(Z_i,N_i)]+(Z_i+N_i)*m_u
            X_i=Xi_time_n_z(N_i,Z_i)
            e=decay_rate_i*(Q_i*X_i*c**2)/m_i
            e_a+=e
            e_a_nuclei_erg.append(e*1e4)
            nuclei_a.append(nuclei)
           
        else: 
            n+=1
    else:
        na+=1

e_a_erg=e_a*1e4
e_b_erg=e_b*1e4
e_a_nuclei_frac=e_a_nuclei_erg/e_a_erg
e_b_nuclei_frac=e_b_nuclei_erg/e_b_erg
print(n,nb,na)

In [ ]:
#find relevant nuclei
time_days=time/(24*60*60)
i_initial=np.searchsorted(time_days, 1)
i_Final=100000

i_s=np.argsort(-np.max(e_b_nuclei_frac[:,i_initial:i_Final], axis=1))
e_b_nuclei_sorted=[e_b_nuclei_erg[i] for i in i_s]
nuclei_b_sorted=[nuclei_b[i] for i in i_s]

i_s=np.argsort(-np.max(e_a_nuclei_frac[:,i_initial:i_Final], axis=1))
e_a_nuclei_sorted=[e_a_nuclei_erg[i] for i in i_s]
nuclei_a_sorted=[nuclei_a[i] for i in i_s]


In [ ]:
#termalization

#important parameters
Mey=0.05
Vey=0.15
time_days=time/(24*60*60)

tb=12.9*((Mey/0.01)**(2/3))*((Vey/0.2)**(-2))*24*60*60 #termalization beta particles
ty=0.3*np.sqrt(Mey/0.01)*(0.2/Vey)*24*60*60 ##termalization gamma particles
f_gamma=1-np.exp(-(ty/time)**2)
f_electrons=(1+time/tb)**(-1)
f_beta=0.2*f_electrons+0.45*f_gamma
f_alpha=(1+time/(3*tb))**(-1)

e_b_ef=e_b_erg*f_beta
e_a_ef=e_a_erg*f_alpha


In [ ]:
'''Preliminar figures'''
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12
})

plt.figure(figsize=(6.5,5.5), dpi=300)
plt.plot(time_days,e_a_ef,label=r'alpha ',color='red')
plt.plot(time_days,e_b_ef,label=r'beta ',color='blue')
plt.plot(time_days,e_a_ef+e_b_ef,label=r'total ', color='black',linestyle='--')
for i in range(6):
    plt.plot(time_days,e_a_nuclei_sorted[i]*f_alpha,label=f'{nuclei_a_sorted[i]}'+r', $\alpha$',linestyle=':')
for i in range(6):
    plt.plot(time_days,e_b_nuclei_sorted[i]*f_beta,label=f'{nuclei_b_sorted[i]}'+r', $\beta$', linestyle='-.')
plt.yscale('log')
plt.xscale('log')
plt.xlabel('Time[Days]')
plt.ylabel(r'Effective heating rate $[erg\,g^{-1}\,s^{-1}]$')
plt.legend(loc="upper right", ncol=2,frameon=False,fontsize=8)
plt.ylim(1e4,1e12)
plt.xlim(0.1,500)
plt.savefig("Run6_Approach_II.pdf", bbox_inches="tight")


In [ ]:

#save results
pd.DataFrame({'Epsilon_alpha':e_a_ef,'Time [Days]':time_days}).to_csv('epsilon_calculations/run6/alpha.csv',index=False)
pd.DataFrame({'Epsilon_beta':e_b_ef,'Time [Days]':time_days}).to_csv('epsilon_calculations/run6/beta.csv',index=False)
for i in range(20):
    pd.DataFrame({'time_days':time_days,f'{nuclei_b_sorted[i]} top {i+1}':e_b_nuclei_sorted[i]}).to_csv(f'epsilon_calculations/run6/b_top/e_r_{i}.csv',index=False)
for i in range(20):
    pd.DataFrame({'time_days':time_days,f'{nuclei_a_sorted[i]} top {i+1}':e_a_nuclei_sorted[i]}).to_csv(f'epsilon_calculations/run6/a_top/e_r_{i}.csv',index=False)
